# Решения: прототип kNN

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from pathlib import Path as _P
_P('figures').mkdir(exist_ok=True)
import matplotlib.pyplot as plt


def show_digit(row, title=''):
    """Одна строка таблицы -> картинка 8x8."""
    values = [int(v) for v in row[PIXELS]]
    grid = [values[i * 8:(i + 1) * 8] for i in range(8)]
    plt.imshow(grid, cmap='gray_r')
    plt.title(title)
    plt.axis('off')


## Урок. 1–3. Подвыборка, baseline, масштаб по train

In [ ]:
sub = df.sample(400, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(sub[PIXELS], sub['label'], test_size=100,
                                          random_state=0, stratify=sub['label'])
top_train = int(y_tr.value_counts().idxmax())
baseline_acc = float((y_te == top_train).mean())
mins = X_tr.min()
rng = (X_tr.max() - X_tr.min()).replace(0, 1)
tr_scaled = (X_tr - mins) / rng
te_scaled = (X_te - mins) / rng
te_max = float(te_scaled.max().max())
LEAK_NOTE = (
    'min/max по всей таблице подсматривают проверочные картинки: настройка препроцессинга '
    'уже зависит от данных, на которых мы обещали честную оценку.'
)
print(top_train, round(baseline_acc, 3), round(te_max, 3))

## Урок. 4–5. Таблица k и время

In [ ]:
rows = []
for k in (1, 3, 5, 7, 9):
    m = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    rows.append([k, float(accuracy_score(y_te, m.predict(X_te)))])
results = pd.DataFrame(rows, columns=['k', 'accuracy'])
best_row = results.sort_values('accuracy', ascending=False).iloc[0]
best_acc, best_k = float(best_row['accuracy']), int(best_row['k'])
import time
small = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr, y_tr)
t0 = time.perf_counter(); small.predict(X_te); t_small = time.perf_counter() - t0
big = KNeighborsClassifier(n_neighbors=best_k).fit(df[PIXELS].iloc[:1200], df['label'].iloc[:1200])
t0 = time.perf_counter(); big.predict(X_te); t_big = time.perf_counter() - t0
print(results, best_k, round(t_small, 4), round(t_big, 4))

## Урок. 6–8. Ошибки, константные пиксели, два признака

In [ ]:
model = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr, y_tr)
pred = model.predict(X_te)
wrong_positions = [i for i in range(len(y_te)) if pred[i] != y_te.iloc[i]]
plt.figure(figsize=(6, 2))
for j, pos in enumerate(wrong_positions[:3]):
    plt.subplot(1, 3, j + 1)
    show_digit(X_te.iloc[pos], title=f'{y_te.iloc[pos]} -> {pred[pos]}')
plt.tight_layout()
errors_png = _P('figures/errors.png'); plt.savefig(errors_png); plt.close()
ERROR_NOTE = 'Ошибки — на смазанных и наклонных начертаниях, где 8/9 и 1/7 похожи по пикселям'
const_pixels = [c for c in PIXELS if df[c].nunique() == 1]
keep = [c for c in PIXELS if c not in const_pixels]
m_nc = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr[keep], y_tr)
acc_no_const = float(accuracy_score(y_te, m_nc.predict(X_te[keep])))
CONST_NOTE = 'Пиксель с одним значением даёт нулевую разность всем: качество не изменилось'
two_tr = pd.DataFrame({'ink': X_tr.sum(axis=1), 'n_dark': (X_tr > 8).sum(axis=1)})
two_te = pd.DataFrame({'ink': X_te.sum(axis=1), 'n_dark': (X_te > 8).sum(axis=1)})
m_two = KNeighborsClassifier(n_neighbors=best_k).fit(two_tr, y_tr)
acc_two = float(accuracy_score(y_te, m_two.predict(two_te)))
FEATURES_NOTE = 'Разные цифры имеют похожую суммарную яркость: два числа не различают форму'
print(len(wrong_positions), round(acc_no_const, 4), round(acc_two, 4))

## ДЗ. 1–4

In [ ]:
sub7 = df.sample(400, random_state=7)
A_tr, A_te, b_tr, b_te = train_test_split(sub7[PIXELS], sub7['label'], test_size=100,
                                          random_state=0, stratify=sub7['label'])
acc_hw = float(accuracy_score(b_te, KNeighborsClassifier(3).fit(A_tr, b_tr).predict(A_te)))
baseline_hw = float((b_te == b_tr.value_counts().idxmax()).mean())
acc_2 = float(accuracy_score(b_te, KNeighborsClassifier(2).fit(A_tr, b_tr).predict(A_te)))
acc_3 = acc_hw
TIE_NOTE = 'При k=2 голоса могут разделиться 1:1, и ответ решает порядок соседей, а не большинство'
rows = []
for n in (50, 100, 200, 300):
    m = KNeighborsClassifier(3).fit(A_tr.iloc[:n], b_tr.iloc[:n])
    rows.append([n, float(accuracy_score(b_te, m.predict(A_te)))])
size_table = pd.DataFrame(rows, columns=['n_train', 'accuracy'])
SIZE_NOTE = ('Точность быстро растёт на первых сотнях примеров и затем почти выходит на плато: '
             'ещё сто картинок дают меньше, чем первые сто.')
csv_path = _P('experiments_hw.csv')
size_table.to_csv(csv_path, index=False)
print(round(acc_hw, 4), round(baseline_hw, 3), round(acc_2, 4), size_table)